<a href="https://colab.research.google.com/github/tas-tasnim/Machine-Learning-Models/blob/main/machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, RandomizedSearchCV,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

df = pd.read_csv('/kaggle/input/titanic/train.csv')
df.pop('Name')
print('Data loading --->> Completed')

Data loading --->> Completed


In [ ]:
y=df.Survived
X=df.drop('Survived', axis=1)

X_train, X_valid , y_train, y_valid = train_test_split(X , y , test_size = 0.2)
print('split completed')

split completed


In [ ]:
#Separating Catagorical & Numerical Columns

objects_col =X_train.select_dtypes(include=['object'])
objects_col_valid =X_valid.select_dtypes(include=['object'])
objects_cols = list(objects_col.columns)
objects_cols_valid = list(objects_col_valid.columns)

num_cols = X_train.select_dtypes(exclude=['object'])
num_cols_valid = X_valid.select_dtypes(exclude=['object'])
num_col = list(num_cols.columns)

numeric_features =num_col
categorical_features = objects_cols

print(objects_cols)
print(num_col)

print('Separation Completed')

['Sex', 'Ticket', 'Cabin', 'Embarked']
['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Separation Completed


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num',numeric_transformer,numeric_features),
        ('cat',categorical_transformer,categorical_features)
    ]
)

print('Pipeline building finish')

Pipeline building finish


In [ ]:
#declaring model
model = RandomForestClassifier(random_state=42)

pipe= Pipeline(steps=[
                    ('preprocessor',preprocessor),
                    ('classifier',model)
])

print('everyting is ok ')

everyting is ok 


In [ ]:
def Get_scores(X_train,y_train,X_valid,y_valid,iters=5,model=RandomForestClassifier()):

    for _ in range(iters):
        #Inputing params
        n_estimators      = int(input("n_estimators :"))
        max_depth         = int(input("max_depth :"))
        min_samples_split = int(input("min_samples_split :"))
        min_samples_leaf  = int(input("min_samples_leaf :"))

        model = RandomForestClassifier(n_estimators=n_estimators
                                , random_state=42,
                                  max_depth= max_depth,
                                  min_samples_split= min_samples_split,
                                  min_samples_leaf= min_samples_leaf,
                                  max_features= "sqrt" )

        pipe.fit(X_train, y_train)

        scores = cross_val_score(pipe, X_train, y_train, cv=5)  # 5-fold CV

        print(f"""\ncross validation : mean : {np.mean(scores)}
    standard derivation : {np.std(scores)} \n""")

        print("Train Accuracy:", pipe.score(X_train, y_train))
        print("Test Accuracy:", pipe.score(X_valid, y_valid))
        print('\n')

print('all ok')

all ok


In [ ]:
def smart_tune(X_train,y_train,X_valid,y_valid,iters=10,model=pipe,mathod=RandomizedSearchCV):
    #declaring parameters--->>>

    params = {
    'classifier__n_estimators': [60,250],
    'classifier__max_depth': [9,13],
    'classifier__min_samples_split': [3,4],
    'classifier__min_samples_leaf': [3, 4]
    }
    i=1
    for _ in range(iters):


        # Searching for best parameter--->>>
        grid = mathod(model,params,cv=3,n_jobs=-1)
        grid.fit(X_train,y_train)
        best_params = grid.best_params_
        Model = grid.best_estimator_

        # Geting best params
        n_estimators      = best_params['classifier__n_estimators']
        max_depth         = best_params['classifier__max_depth']
        min_samples_split = best_params['classifier__min_samples_split']
        min_samples_leaf  = best_params['classifier__min_samples_leaf']

        #Printing Best_params
        print(f"""\niter = {i}
        best_params -->>
                 n_estimators       : {best_params['classifier__n_estimators']}
                 max_depth          : {best_params['classifier__max_depth']}
                 min_samples_split  : {best_params['classifier__min_samples_split']}
                 min_samples_leaf   : {best_params['classifier__min_samples_leaf']}\n""")
        #cross_val_score --->>

        scores = cross_val_score(Model, X_train, y_train, cv=5)  # 5-fold CV

        print(f"""cross validation : mean : {np.mean(scores)}
    standard derivation : {np.std(scores)} \n""")

        print("Train Accuracy:", Model.score(X_train, y_train))
        print("Test Accuracy:", Model.score(X_valid, y_valid))
        # Sorting inside the params
        for key in params:
            params[key].sort()
        i+=1
        # n_estimator re-generate
        if params['classifier__n_estimators'].index(n_estimators) == 1:
            params['classifier__n_estimators'][0] = n_estimators + 30
        else:
             params['classifier__n_estimators'][1] = n_estimators - 30

        # max_depth re-generate
        if params['classifier__max_depth'].index(max_depth) == 1:
            params['classifier__max_depth'][0] = max_depth + 1
        else:
            params['classifier__max_depth'][1] = max_depth - 1

        # min_samples_split re-generate
        if params['classifier__min_samples_split'].index(min_samples_split) == 1:
            params['classifier__min_samples_split'][0] = min_samples_split + 1
        else:
            params['classifier__min_samples_split'][1] = min_samples_split - 1

        # max_depth re-generate
        if params['classifier__min_samples_leaf'].index(min_samples_leaf) == 1:
            params['classifier__min_samples_leaf'][0] = min_samples_leaf + 1
        else:
            params['classifier__min_samples_leaf'][1] = min_samples_leaf - 1

    return best_params

print('Function is ready!')

Function is ready!


In [ ]:
#Trainning model

pipe.fit(X_train,y_train)

y_pred = pipe.predict(X_valid)



Get_scores(X_train,y_train,X_valid,y_valid,5,pipe)

n_estimators : 200
max_depth : 50
min_samples_split : 3
min_samples_leaf : 7



cross validation : mean : 0.8189008174923668 
    standard derivation : 0.03599569658453297 

Train Accuracy: 1.0
Test Accuracy: 0.8603351955307262




KeyboardInterrupt: Interrupted by user

In [ ]:
smart_tune(X_train,y_train,X_valid,y_valid,10,pipe)


iter = 1
        best_params -->> 
                 n_estimators       : 250
                 max_depth          : 13
                 min_samples_split  : 3
                 min_samples_leaf   : 3

cross validation : mean : 0.6263764404609475 
    standard derivation : 0.012030772608298337 

Train Accuracy: 0.6320224719101124
Test Accuracy: 0.5977653631284916

iter = 2
        best_params -->> 
                 n_estimators       : 250
                 max_depth          : 13
                 min_samples_split  : 3
                 min_samples_leaf   : 2

cross validation : mean : 0.7471584753274894 
    standard derivation : 0.036449968740800504 

Train Accuracy: 0.8019662921348315
Test Accuracy: 0.8100558659217877

iter = 3
        best_params -->> 
                 n_estimators       : 220
                 max_depth          : 12
                 min_samples_split  : 4
                 min_samples_leaf   : 1

cross validation : mean : 0.8131783709248499 
    standard derivation : 

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
12 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py", line 340, in fit
    self._validate_params()
  File "/usr/local/lib/python3.11/dist-packag


iter = 4
        best_params -->> 
                 n_estimators       : 220
                 max_depth          : 12
                 min_samples_split  : 4
                 min_samples_leaf   : 1

cross validation : mean : 0.8131783709248499 
    standard derivation : 0.014721025368397762 

Train Accuracy: 0.8848314606741573
Test Accuracy: 0.8547486033519553

iter = 5
        best_params -->> 
                 n_estimators       : 250
                 max_depth          : 13
                 min_samples_split  : 4
                 min_samples_leaf   : 1

cross validation : mean : 0.8117994681374963 
    standard derivation : 0.016836799559402857 

Train Accuracy: 0.8848314606741573
Test Accuracy: 0.8603351955307262


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
15 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py", line 340, in fit
    self._validate_params()
  File "/usr/local/lib/python3.11/dist-packag


iter = 6
        best_params -->> 
                 n_estimators       : 250
                 max_depth          : 13
                 min_samples_split  : 4
                 min_samples_leaf   : 1

cross validation : mean : 0.8117994681374963 
    standard derivation : 0.016836799559402857 

Train Accuracy: 0.8848314606741573
Test Accuracy: 0.8603351955307262

iter = 7
        best_params -->> 
                 n_estimators       : 220
                 max_depth          : 12
                 min_samples_split  : 3
                 min_samples_leaf   : 1

cross validation : mean : 0.8131783709248499 
    standard derivation : 0.013346903039090254 

Train Accuracy: 0.8806179775280899
Test Accuracy: 0.8435754189944135


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
15 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py", line 340, in fit
    self._validate_params()
  File "/usr/local/lib/python3.11/dist-packag


iter = 8
        best_params -->> 
                 n_estimators       : 220
                 max_depth          : 12
                 min_samples_split  : 3
                 min_samples_leaf   : 1

cross validation : mean : 0.8131783709248499 
    standard derivation : 0.013346903039090254 

Train Accuracy: 0.8806179775280899
Test Accuracy: 0.8435754189944135

iter = 9
        best_params -->> 
                 n_estimators       : 220
                 max_depth          : 12
                 min_samples_split  : 4
                 min_samples_leaf   : 1

cross validation : mean : 0.8131783709248499 
    standard derivation : 0.014721025368397762 

Train Accuracy: 0.8848314606741573
Test Accuracy: 0.8547486033519553


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
18 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
18 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py", line 340, in fit
    self._validate_params()
  File "/usr/local/lib/python3.11/dist-packag


iter = 10
        best_params -->> 
                 n_estimators       : 190
                 max_depth          : 12
                 min_samples_split  : 5
                 min_samples_leaf   : 1

cross validation : mean : 0.8033389146065202 
    standard derivation : 0.019662618550698775 

Train Accuracy: 0.8806179775280899
Test Accuracy: 0.8547486033519553


{'classifier__n_estimators': 190,
 'classifier__min_samples_split': 5,
 'classifier__min_samples_leaf': 1,
 'classifier__max_depth': 12}